In [7]:
import numpy as np
import pandas as pd

In [8]:
books = pd.read_csv("books.csv")[["book_id", "title", "authors"]]
ratings = pd.read_csv("ratings.csv")

In [9]:
ratings = ratings.merge(books[["book_id", "title"]], how="left", on="book_id").dropna()

In [10]:
uii_matrix = ratings.pivot_table(index=["user_id"], columns=["title"], values="rating").fillna(np.nan)

In [20]:
def similarity(user_id):
    min_common_rated = 10
    my_books_read = uii_matrix.loc[user_id].notna()
    intersections = uii_matrix.apply(lambda x: (x.notna() & my_books_read).sum(), axis=1)
    similarities = uii_matrix[intersections >= min_common_rated].corrwith(uii_matrix.loc[user_id], axis=1)
    similarities[user_id] = np.nan
    return similarities

In [21]:
def scoring(column, user_similarities):
    sim = user_similarities.reindex(column.index)
    neighbours = sim > min_similarity
    valid = neighbours & column.notna()
    numerator = np.sum(column[valid] * sim[valid])
    denominator = np.sum(sim[valid])
    predicted_rating = numerator / denominator if denominator != 0 else np.nan
    if valid.sum() <= min_ratings:
        predicted_rating = np.nan
    return predicted_rating

In [22]:
min_similarity = 0.7
min_ratings = 5
scores = uii_matrix.apply(lambda x: scoring(x, similarity(1)))

/Users/firdavs/PycharmProjects/BookRecommendationSystem/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/firdavs/PycharmProjects/BookRecommendationSystem/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


KeyboardInterrupt: 

In [19]:
scores[~my_books_read].sort_values(ascending=False)

title
Lonesome Dove                                                                  4.884893
Ahab's Wife, or The Star-Gazer                                                 4.852697
Someone Knows My Name                                                          4.844823
The Diving Bell and the Butterfly                                              4.828780
A Storm of Swords: Blood and Gold (A Song of Ice and Fire, #3: Part 2 of 2)    4.810564
                                                                                 ...   
واحة الغروب                                                                         NaN
يوتوبيا                                                                             NaN
ڤيرتيجو                                                                             NaN
キスよりも早く1 [Kisu Yorimo Hayaku 1] (Faster than a Kiss #1)                             NaN
美少女戦士セーラームーン新装版 1 [Bishōjo Senshi Sailor Moon Shinsōban 1]                          NaN
Length: 9904, dtype: float